In [9]:
# NOTE: this script is designed to be run with the "arcgis_env" python environment which is simply the ArcGIS Pro python environment cloned in Anaconda.
# NOTE: if the script crashes, delete the layers that correspond to the sea level that was being processed when the crash occurred and delete everything in the temp folder. Then re-run the script.

# import packages
import numpy as np
import arcpy
import sys, datetime
from arcpy.sa import *
from arcpy.ia import *
import scipy
import scipy.special as ss
import os
import gc
import shutil

In [10]:
# Initialize ArcPy
arcpy.CheckOutExtension("Spatial") # check out Spatial Analyst
arcpy.env.overwriteOutput=True # enable overwriting existing outputs

# Set Environments
arcpy.env.workspace = r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/CRC_Static_SLR/CRC_Static_SLR.gdb" # path to your geodatabase
arcpy.env.cellSize = "MINOF" # set cell size to the minimum of all rasters
arcpy.env.extent = "MINOF" # set extent to the minimum of all rasters

# directories to save files
temp_path = r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/temp/" # path to your temp folder

# Create coastline

In [6]:
## NOTE: DO NOT RUN THIS CELL IF YOU ALREADY HAVE A COASTLINE SHAPEFILE

coast_save = r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/" # path to save your coastline shapefile

# ---------------------- Set Variables ---------------------- #
SLR = 0.02 # 0.02 or 0.04 m to account for displacement of MHHW from 1992 to 2000 or 2005, respectively, as stated by NOAA.
name = 'MHHW' # name of sea level at which you will create the coastline. Your file will have this string attached in the name
# islands = ['Oahu','Kauai','Lanai','Maui','Molokai','Hawaii','Kahoolawe','Niihau'] # name of the island you are working on. This is used to create the output file name. You can add more islands by adding them to the list, but make sure you have the corresponding DEM files in the input folder.
islands = ['Kahoolawe']

In [ ]:
# ---------------------- Create Coastline ---------------------- 

now = datetime.datetime.now() # check start time
print("Ua ho'omaka i ka hola " + now.strftime('%H:%M:%S')) # print start time

# set water level and define parameters
tcari = Raster(r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/HI_TCARI_EPSG6634/Hawaii_TCARI_MHHW_EPSG6634_500m_LMSLm.tif") 
water_uncert = 0.073 # std with 1 ddof of the difference between MHHW and MSL in meters.

for island in islands:
    # load rasters
    
    dem = Raster(f"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/{island}/HI_{island}_LMSLm_EPSG6634.tif") # path to the DEM file
    arcpy.env.cellSize = dem
    # set water level and define parameters
    waterSurface = tcari + SLR # add sea level to the TCARI surface

    if island== 'Oahu':
        dem_uncer = 0.3

    elif island== 'Kahoolawe':
        dem_uncer = 0.62

    elif island== 'Niihau':
        dem_uncer = 0.56

    else:
        dem_uncer = 0.2
    # NOTE:  
    # - The uncertainty value we used for all islands, except Oahu, Kahoolawe, and Niihau is 0.2 m.    
    # - Talk to Matt to figure this out if necessary.
    
    sigma_d = np.sqrt(dem_uncer**2+water_uncert**2) # cummulative uncertainty
    prob = 0.8 # probability of inundaiton (i.e., confidence of a given area being wet; 0.8 = 80%)

    # calculate SLR inundation
    u_dem = (waterSurface - dem) + sigma_d*np.sqrt(2)*ss.erfcinv(2*prob) # this is where the magic happens.
    # the formula above looks at every pixel and calculates the inundaiton at that pixel based on the uncertainty of the DEM and the probability of inundation.

    # identify total flood 
    if island == 'Kahoolawe':
        reclass_totalflood= Con(IsNull(u_dem), 1, Con(u_dem >= 0, 1)) # reclassify the raster to identify flooded areas. For some reason I have to run Kahoolawe with this version of reclass_totalflood. I suspect is has something to do with the DEM reaching the edge of the raster extent, but didn't really look into it.
    else:
        reclass_totalflood = Con(dem <= u_dem, 1) # This version of reclass_totalflood works for all islands except Kahoolawe.

    # All negative is dry, all positive is wet.
    region_grp = RegionGroup(reclass_totalflood, "EIGHT", "CROSS", "", "") # select cluster of pixels with 8 neighbors to identify ocean connectivity.
    region_grp_path = f"{temp_path}{island}_{str(int(prob*100))}prob_RegGrp.tif"
    region_grp.save(region_grp_path) # save the raster

    # find region group value given to the ocean
    arcpy.management.BuildRasterAttributeTable(region_grp_path, "OVERWRITE")    # make sure the raster attribute table exists
    max_region = None
    max_count = -1
    with arcpy.da.SearchCursor(region_grp_path, ["Value", "Count"]) as cursor:
        for val, cnt in cursor:
            if cnt > max_count:
                max_count = cnt # number of pixels with a given region group value
                max_region = val # region group value

    # identify conectivity to ocean
    region_grp_path = Raster(region_grp_path)
    oceanOnlySelect = Con(region_grp_path == max_region, 1, 2) # assign value = 1 to the ocean connected region, value = 2 to all other regions. If you want to automatically delete everything that is not ocean, close parenthesis after "...max_region, 1" and write "#"

    oceanOnlySelect.save(f"{temp_path}{island}_{str(int(prob*100))}prob_RegGrp_Reclass_{name}.tif") # save the raster
    oceanOnlySelect_polygName = f"{coast_save}{island}/{island}_{str(int(prob*100))}prob_{name}_coastline.shp" # name of the shapefile to save the coastline
    arcpy.RasterToPolygon_conversion(oceanOnlySelect, oceanOnlySelect_polygName,"NO_SIMPLIFY","Value", "MULTIPLE_OUTER_PART") # conversion of raster to polygon of coastline
    arcpy.Delete_management(oceanOnlySelect)
    arcpy.Delete_management(region_grp_path)

    now = datetime.datetime.now()
    print(("Ua pau ka mea o " + island + " i ka hola " + now.strftime('%H:%M:%S'))) # print end time of island
print("Ua pau na mea a pau")

Ua ho'omaka i ka hola 15:12:06
Ua pau ka mea o Kahoolawe i ka hola 15:13:22
Ua pau na mea a pau


# Passive SLR Modeling

In [11]:
# ---------------------- Set Variables ---------------------- #

# List of MHHW surfaces for area, will loop through
# SLR = [0.04, 0.3448, 0.6496, 0.9544, 1.2592, 1.564, 1.8688, 2.1736, 2.4784, 2.7832, 3.088] # SLR increments from 0-10ft in meters + 0.04 m to account for displacement of MHHW from 1992 to 2005 as stated by NOAA
SLR = [0.02, 0.3248, 0.6296, 0.9344, 1.2392, 1.544, 1.8488, 2.1536, 2.4584, 2.7632, 3.068] # SLR increments from 0-10ft in meters + 0.02 m to account for displacement of MHHW from 1992 to 2000 as stated by NOAA

name = ['00ft', '01ft', '02ft', '03ft', '04ft','05ft','06ft','07ft','08ft','09ft','10ft']
islands = ['Kauai','Lanai','Maui','Molokai','Hawaii','Kahoolawe','Niihau']
           
# set water level and define parameters
tcari = Raster(r"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/HI_TCARI_EPSG6634/Hawaii_TCARI_MHHW_EPSG6634_500m_LMSLm.tif") 
water_uncert = 0.073 # std with 1 ddof of the difference between MHHW and MSL in meters.

prob = 0.8
prob_name = str(int(prob*100))+'prob'
reclass_ranges = [[-10000,0,'NoData'],[0,0.3048,1],[0.3048001,0.6096,2],[0.6096001,0.9144,3],[0.9144001,1.2192,4],[1.2192001,1.524,5], [1.524001,1.8288,6],[1.8288001,2.1336,7],[2.1336001,2.4384,8],[2.4384001,2.7432,9],[2.7432001,3.048,10],[3.048001,10000,11]]

In [13]:
now = datetime.datetime.now()
print(f"Ua hoomaka i ka hola {now.strftime('%H:%M:%S')}")

for island in islands:
    now = datetime.datetime.now()
    print(f"Ua hoomaka ka modeling o {island} i ka hola {now.strftime('%H:%M:%S')}")

    dem = Raster(f"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/{island}/HI_{island}_LMSLm_EPSG6634.tif") # path to the DEM file
    if island== 'Kahoolawe':
        dem_uncer = 0.62
    elif island== 'Niihau':
        dem_uncer = 0.56
    else:
        dem_uncer = 0.2 # NOTE: If no value or unsure if its correct, alk to Matt to figure this out.

    sigma_d = np.sqrt(dem_uncer**2+water_uncert**2) # cummulative uncertainty

    arcpy.env.cellSize = dem
    arcpy.env.extent = dem
    arcpy.env.snapRaster = dem
    
    coastline = f"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Inputs/{island}/{island}_{prob_name}_MHHW_coastline.shp"
    save_path = f"C:/Users/npaoakan/Desktop/CRC_Static_SLR/Outputs/{island}/"

    for i, slr in enumerate(SLR):
        now = datetime.datetime.now()
        print(f"Ke hana nei i ka mea {name[i]} i ka hola {now.strftime('%H:%M:%S')}")

        waterSurface = tcari + slr # start with 0 ft SLR

        d_tilda = (waterSurface - dem) + sigma_d*np.sqrt(2)*ss.erfcinv(2*prob) # this is where the magic happens.
        # print('pau with d_tilda')
        # create shapefile of total flood
        reclass_totalflood = Con(d_tilda >= 0, 1)
        # print('pau with reclass_totalflood')
        region_grp = RegionGroup(reclass_totalflood, "EIGHT", "CROSS", "", "") # select cluster of pixels with 8 neighbors to identify ocean connectivity.
        # print('pau with region_grp')
        region_grp_path = f"{temp_path}{island}_{prob_name}_{name[i]}_RegGrp.tif"
        region_grp.save(region_grp_path) # save the raster
        # print('pau with region_grp_save')
        oceanOnlySelectPoly_path = f"{temp_path}{island}_{prob_name}_RegGrpPoly_Reclass_{name[i]}.shp"
        region_reclass_totalflood = arcpy.RasterToPolygon_conversion(region_grp_path,
                                                                     oceanOnlySelectPoly_path,
                                                                     "NO_SIMPLIFY",
                                                                     "Value",
                                                                     "MULTIPLE_OUTER_PART")
        # Topographically Isolated Inundation (TII)
        arcpy.MakeFeatureLayer_management(region_reclass_totalflood,
                                          'tempName') # 'tempName' is just a placeholder for the shapefile
        arcpy.SelectLayerByLocation_management('tempName',
                                                "INTERSECT",
                                                coastline,
                                                None,
                                                "NEW_SELECTION", "INVERT") # select the features that are not intersecting the shoreline
        # in other words, select features that are topographically isolated
        ti_inundation_shp = f"{temp_path}tiI_{island}_{prob_name}_{name[i]}.shp"
        arcpy.CopyFeatures_management('tempName',
                                        ti_inundation_shp) # save topographically isolated features to a shapefile
        arcpy.DeleteFeatures_management('tempName') # delete topographically isolated features from the original shapefile

        arcpy.AddGeometryAttributes_management(ti_inundation_shp,
                                               "AREA",
                                               Area_Unit="SQUARE_METERS") # calculate area of each polygon in the shapefile
        arcpy.MakeFeatureLayer_management(ti_inundation_shp,
                                          'tempName_area') # 'tempName_area' is just a placeholder for the shapefile. Needed for selecting features
        
        arcpy.SelectLayerByAttribute_management('tempName_area',
                                                "NEW_SELECTION",
                                                "POLY_AREA<=100") # select polygons with area less than 100 m^2. Enforcing all computations with an area rather than pixel number.

        arcpy.DeleteFeatures_management('tempName_area') # delete all polygons selected polygons
        print('pau with DeleteFeatures_management for area')
        
        with arcpy.da.SearchCursor(ti_inundation_shp, "*") as cursor:
            features = [row for row in cursor] # check if there are any areas left after deleting small clusters of pixels

        if not features: # if there are no TII areas then...
            print('no TII areas')

            if name[i] == '00ft': # if name is 0p0m (which also means that there is no SCI inundation because at 0.0m of SLR the flood = the coastline) then there's no layer created
                print("a'ohe mea i loa'a mai a a'ohe flood map o ka SLR increment 0.0m")
                pass

            else: # if its any increment other than 0p0m then there is SCI inundation...
                arcpy.RepairGeometry_management(in_features=oceanOnlySelectPoly_path,
                                                delete_null="KEEP_NULL",
                                                validation_method="ESRI")
                sc_inundation_shp = f"{temp_path}{island}_{prob_name}_{name[i]}_SCI.shp"
                arcpy.Erase_analysis(oceanOnlySelectPoly_path, coastline, sc_inundation_shp, '#') # erase portion of sc inundation shapefile that is oceanwards from the coastline
                
                sc_inundation_raster = f"{save_path}SCI/Unclass/{island}_{prob_name}_{name[i]}_SCI.tif"
                arcpy.Clip_management(d_tilda,
                                      "#",
                                      sc_inundation_raster,
                                      sc_inundation_shp,
                                      "#",
                                      "ClippingGeometry",
                                      "NO_MAINTAIN_EXTENT") # clip the total flood raster to create a raster of surficially connected inundation
                # reclassify surficially connected inundation to ranges of flood depth
                sc_inundation_raster_reclass = Reclassify(sc_inundation_raster,
                                                          "VALUE",
                                                          RemapRange(reclass_ranges))
                sc_inundation_raster_reclass_path = f"{temp_path}{island}_{name[i]}_SCI.tif"
                sc_inundation_raster_reclass.save(sc_inundation_raster_reclass_path) # save the reclassified raster to the temp file because it will saved with a high bit depth
                new_sc_inundation_raster_reclass = f"{save_path}SCI/{island}_{prob_name}_{name[i]}_SCI.tif"
                arcpy.CopyRaster_management(sc_inundation_raster_reclass_path, 
                                            new_sc_inundation_raster_reclass, 
                                            "", 
                                            "", 
                                            "NoData", 
                                            "NONE", 
                                            "NONE", 
                                            "4_BIT") # change bit depth and save to the final location
                arcpy.Delete_management(sc_inundation_raster_reclass_path) # delete high bit raster to save space
                arcpy.Delete_management(sc_inundation_raster_reclass) # delete the original raster to save space
                arcpy.BuildRasterAttributeTable_management(new_sc_inundation_raster_reclass, "Overwrite")
                del sc_inundation_raster_reclass

                # total inundation is only SCI inundation because there is no TII inundation
                arcpy.Copy_management(sc_inundation_shp, f"{save_path}Extent_shp/{island}_{prob_name}_{name[i]}_totalI.shp")

        else: # if there are TII areas then...
            print('get TII areas')
            if name[i] == '00ft': # if name is 0p0m (which also means that there is no SCI inundation because at 0.0m of SLR the flood = the coastline) then the TII is also the total inundation...
                temp_cl_ti_inundation_raster = f"{temp_path}{island}_{prob_name}_{name[i]}_TII.tif"
                arcpy.Clip_management(d_tilda,
                                      "#",
                                      temp_cl_ti_inundation_raster,
                                      ti_inundation_shp,
                                      "#",
                                      "ClippingGeometry",
                                      "NO_MAINTAIN_EXTENT") # use the shapefile to clip the raster of topographically isolated inundation so that there are no areas with less than 100 pixels
                # reclassify surficially connected inundation to ranges of flood depth
                ti_inundation_raster_reclass = Reclassify(temp_cl_ti_inundation_raster,
                                                          "VALUE",
                                                          RemapRange(reclass_ranges))
                ti_inundation_raster_reclass_path = f"{temp_path}{island}_{prob_name}_{name[i]}_TII.tif"
                ti_inundation_raster_reclass.save(ti_inundation_raster_reclass_path)

                # the clip tool will create a raster with a high bit depth, that is why is saved in the temp folder
                cl_ti_inundation_raster = f"{save_path}TII/{island}_{prob_name}_{name[i]}_TII.tif"
                arcpy.CopyRaster_management(ti_inundation_raster_reclass_path, 
                                            cl_ti_inundation_raster, 
                                            "", 
                                            "", 
                                            "NoData", 
                                            "NONE", 
                                            "NONE", 
                                            "4_BIT") # copy the temporary topographically isolated inundation raster with a lower bit depth
                arcpy.Delete_management(temp_cl_ti_inundation_raster) # delete the temporary raster to save space
                arcpy.BuildRasterAttributeTable_management(cl_ti_inundation_raster, "Overwrite")

                arcpy.Copy_management(ti_inundation_shp, f"{save_path}Extent_shp/{island}_{prob_name}_{name[i]}_totalI.shp")
                arcpy.Delete_management(ti_inundation_raster_reclass)
                del ti_inundation_raster_reclass

            else: # for all other increments (not 0p0m), there is both TII and SCI inundation so we create the total inundation shapefile with both TII and SCI
                temp_cl_ti_inundation_raster = f"{temp_path}{island}_{prob_name}_{name[i]}_TII.tif"
                arcpy.Clip_management(d_tilda,
                                      "#",
                                      temp_cl_ti_inundation_raster,
                                      ti_inundation_shp,
                                      "#",
                                      "ClippingGeometry",
                                      "NO_MAINTAIN_EXTENT") # use the shapefile to clip the raster of topographically isolated inundation so that there are no areas with less than 100 pixels
                # reclassify surficially connected inundation to ranges of flood depth
                ti_inundation_raster_reclass = Reclassify(temp_cl_ti_inundation_raster,
                                                          "VALUE",
                                                          RemapRange(reclass_ranges))
                ti_inundation_raster_reclass_path = f"{temp_path}{island}_{prob_name}_{name[i]}_TII.tif"
                ti_inundation_raster_reclass.save(ti_inundation_raster_reclass_path)

                # the clip tool will create a raster with a high bit depth, that is why is saved in the temp folder
                cl_ti_inundation_raster = f"{save_path}TII/{island}_{prob_name}_{name[i]}_TII.tif"
                arcpy.CopyRaster_management(ti_inundation_raster_reclass_path, 
                                            cl_ti_inundation_raster, 
                                            "", 
                                            "", 
                                            "NoData", 
                                            "NONE", 
                                            "NONE", 
                                            "4_BIT") # copy the temporary topographically isolated inundation raster with a lower bit depth
                arcpy.Delete_management(temp_cl_ti_inundation_raster) # delete the temporary raster to save space
                arcpy.BuildRasterAttributeTable_management(cl_ti_inundation_raster, "Overwrite")


                # Surficially Connected Inundation (SCI)
                arcpy.RepairGeometry_management(in_features=oceanOnlySelectPoly_path,
                                                delete_null="KEEP_NULL",
                                                validation_method="ESRI")
                sc_inundation_shp = f"{temp_path}{island}_{prob_name}_{name[i]}_SCI.shp"
                arcpy.Erase_analysis(oceanOnlySelectPoly_path, coastline, sc_inundation_shp, '#') # erase portion of sc inundation shapefile that is oceanwards from the coastline
                
                sc_inundation_raster = f"{save_path}SCI/Unclass/{island}_{prob_name}_{name[i]}_SCI.tif"
                arcpy.Clip_management(d_tilda,
                                      "#",
                                      sc_inundation_raster,
                                      sc_inundation_shp,
                                      "#",
                                      "ClippingGeometry",
                                      "NO_MAINTAIN_EXTENT") # clip the total flood raster to create a raster of surficially connected inundation
                # reclassify surficially connected inundation to ranges of flood depth
                sc_inundation_raster_reclass_path = f"{temp_path}{island}_{name[i]}_SCI.tif"
                sc_inundation_raster_reclass = Reclassify(sc_inundation_raster,
                                                          "VALUE",
                                                          RemapRange(reclass_ranges)) # save the reclassified raster to the temp file because it will saved with a high bit depth

                
                sc_inundation_raster_reclass.save(sc_inundation_raster_reclass_path)
                new_sc_inundation_raster_reclass = f"{save_path}SCI/{island}_{prob_name}_{name[i]}_SCI.tif"
                arcpy.CopyRaster_management(sc_inundation_raster_reclass_path, 
                                            new_sc_inundation_raster_reclass, 
                                            "", 
                                            "", 
                                            "NoData", 
                                            "NONE", 
                                            "NONE", 
                                            "4_BIT")
                
                # total inundation shapefile is created by merging the TII and SCI shapefiles
                arcpy.Merge_management([ti_inundation_shp, sc_inundation_shp], f"{save_path}Extent_shp/{island}_{prob_name}_{name[i]}_totalI.shp")

                arcpy.Delete_management(sc_inundation_raster_reclass_path) # delete high bit raster to save space
                arcpy.Delete_management(sc_inundation_raster_reclass) # delete the original raster to save space
                arcpy.BuildRasterAttributeTable_management(new_sc_inundation_raster_reclass, "Overwrite")
                del sc_inundation_raster_reclass


                arcpy.Delete_management(ti_inundation_raster_reclass)
                del ti_inundation_raster_reclass
        
        # arcpy.management.Delete(total_ti_inundation_raster) # delete the raster that contains the pixel clusters (we only want the one that doesnt have small clusters)
        # arcpy.Delete_management(region_grp_ti_inundation_shp_path)

        if name[i] != '00ft':
            del (sc_inundation_shp, sc_inundation_raster)

        # Force garbage collection and clear ArcPy cache
        arcpy.ClearWorkspaceCache_management()
        arcpy.ClearEnvironment("workspace")
        gc.collect()

        # empty temp folder. I do this because sometimes the script crashes when there are too many files in the temp folder.
        for f in os.listdir(temp_path):
            fp = os.path.join(temp_path, f)
            try:
                if os.path.isfile(fp):
                    os.remove(fp)
            except Exception as e:
                print(f"Could not delete {fp}: {e}")
                
        now = datetime.datetime.now()
        print(f"Pau me ka mea {name[i]} o {island} i ka hola {now.strftime('%H:%M:%S')}")
print('Pau me na mea apau')

Ua hoomaka i ka hola 13:40:45
Ua hoomaka ka modeling o Kauai i ka hola 13:40:45
Ke hana nei i ka mea 00ft i ka hola 13:40:45
pau with DeleteFeatures_management for area
get TII areas
Could not delete C:/Users/npaoakan/Desktop/CRC_Static_SLR/temp/Kauai_80prob_00ft_RegGrp.tif: [WinError 32] The process cannot access the file because it is being used by another process: 'C:/Users/npaoakan/Desktop/CRC_Static_SLR/temp/Kauai_80prob_00ft_RegGrp.tif'
Could not delete C:/Users/npaoakan/Desktop/CRC_Static_SLR/temp/Kauai_80prob_RegGrpPoly_Reclass_00ft.shp.HELE.32964.2456.sr.lock: [WinError 32] The process cannot access the file because it is being used by another process: 'C:/Users/npaoakan/Desktop/CRC_Static_SLR/temp/Kauai_80prob_RegGrpPoly_Reclass_00ft.shp.HELE.32964.2456.sr.lock'
Could not delete C:/Users/npaoakan/Desktop/CRC_Static_SLR/temp/tiI_Kauai_80prob_00ft.shp.HELE.32964.2456.sr.lock: [WinError 32] The process cannot access the file because it is being used by another process: 'C:/Users